# Foodborne example

Transcriptome Analysis of *Listeria monocytogenes* Exposed to Beef Fat Reveals Antimicrobial and Pathogenicity Attenuation Mechanisms

**doi**: https://doi.org/10.1128/AEM.03027-20

Ref: Chen YY, Kommadath A, Vahmani P, Visvalingam J, Dugan MER, Yang X. 2021. Transcriptome Analysis of Listeria monocytogenes Exposed to Beef Fat Reveals Antimicrobial and Pathogenicity Attenuation Mechanisms. Appl Environ Microbiol 87:e03027-20.
https://doi.org/10.1128/AEM.03027-20

- The example was done just for C18:2n-6

Obj: extract background and upregulated genes for functional enrichment analysis

## Setup and inputs

Run from `Examples/Foodborne bacteria/`. Set `SOURCE_FILE` below to the original
study spreadsheet (not bundled with this repository). The required columns are
`Gene_id`, `gene_name`, `C18:2n-6_fold_change`, `C18:2n-6_pValue`, and
`C18:2n-6_fdr`. Fold changes are ratios, not log2 values. This notebook builds
`df_all.xlsx` and all inputs for `Analysis.ipynb`; no prepared spreadsheet is
required. KEGG lookups require internet access. Review `unmapped_genes.xlsx`
before interpreting the analysis. Saved outputs were cleared after the fixes.


In [ ]:
from pathlib import Path
import pandas as pd

SOURCE_FILE = Path("aem.03027-20-s0002.xlsx")
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(
        f"Provide the original study spreadsheet at {SOURCE_FILE.resolve()} "
        "or update SOURCE_FILE. See README.md in this directory."
    )
df = pd.read_excel(SOURCE_FILE)
df.head()


In [ ]:
#select the columns of interest
columns_to_keep = ['Gene_id',
    "gene_name",
    "C18:2n-6_fold_change",
    "C18:2n-6_pValue",
    "C18:2n-6_fdr"
]
df = df[columns_to_keep]
df

### KEGG ID

In [ ]:
from ResPathExplorer.mapper_KeggFunctions import search_gene_id_kegg

In [ ]:
import re
from functools import lru_cache


@lru_cache(maxsize=None)
def lookup_gene_name(gene_name):
    return search_gene_id_kegg(gene_name, org_code="lmo")


def resolve_kegg_id(row):
    for column in ("Gene_id", "gene_name"):
        value = row[column]
        if pd.isna(value):
            continue
        candidate = str(value).strip()
        if re.fullmatch(r"(?:lmo:)?lmo\d{4}", candidate, flags=re.IGNORECASE):
            return "lmo:" + candidate.split(":")[-1].lower()
    if pd.isna(row["gene_name"]) or not str(row["gene_name"]).strip():
        return pd.NA
    return lookup_gene_name(str(row["gene_name"]).strip()) or pd.NA


df_all = df.copy()
df_all["KEGG_ID"] = df_all.apply(resolve_kegg_id, axis=1)


In [ ]:
unmapped_genes = df_all[df_all["KEGG_ID"].isna()].copy()
unmapped_genes.to_excel("unmapped_genes.xlsx", index=False)
print(f"Unmapped genes excluded from KEGG analysis: {len(unmapped_genes)} / {len(df_all)}")
unmapped_genes.head()


### Background genes

All genes listed

In [ ]:
df_all.to_excel("df_all.xlsx", index=False)
df_all.head()


In [ ]:
columns_to_keep = ['Gene_id',
    "gene_name",
    "C18:2n-6_fold_change",
    "C18:2n-6_pValue",
    "C18:2n-6_fdr",
    "KEGG_ID"
]
df_all = df_all[columns_to_keep]
df_all.head()

In [ ]:
duplicates_b = df_all[df_all.duplicated(subset='KEGG_ID', keep=False)]
duplicates_b

In [ ]:
df_all = (
    df_all.dropna(subset=['KEGG_ID'])  # Remove linhas com NA em KEGG_ID
          .sort_values(
              by=["KEGG_ID", "C18:2n-6_fold_change", "C18:2n-6_fdr"],
              ascending=[True, False, True]
          )
          .drop_duplicates(subset='KEGG_ID', keep='first')
)

In [ ]:
duplicates_b = df_all[df_all.duplicated(subset='KEGG_ID', keep=False)]
duplicates_b

In [ ]:
background = [x.split(":")[-1] for x in df_all["KEGG_ID"]]
# background

In [ ]:
with open('background.txt', 'w') as file:
    for gene in background:
        file.write(f"{gene}\n")

In [ ]:
len(background)

### Upregulated genes

All genes IDs with fold change >= 2.5 (use in the study) and FDR <= 0.05

In [ ]:
df_up = df_all[
    (df_all["C18:2n-6_fold_change"] >= 2.5) &
    (df_all["C18:2n-6_fdr"] <= 0.05)
].copy()
df_up.head()

In [ ]:
upregulted = [x.split(":")[-1] for x in df_up["KEGG_ID"]]
with open('upregulated.txt', 'w') as file:
    for gene in upregulted:
        file.write(f"{gene}\n")
        
len(upregulted)

#### Gene names

In [ ]:
from ResPathExplorer.mapper_KeggFunctions import get_gene_name_by_kegg_id

dict_g = {}
not_found_list = []
for id_g in upregulted:
    name = get_gene_name_by_kegg_id("lmo:" + id_g)
    if name:
        dict_g[id_g] = name.split(",", 1)[0].strip()
    else:
        not_found_list.append(id_g)
print(f"Genes without a KEGG symbol: {len(not_found_list)}")


In [ ]:
filtered_dict_g = {k: v for k, v in dict_g.items() if v != ""}
len(filtered_dict_g)

In [ ]:
with open("upregulatedGNames.txt", "w") as f:
    for v in filtered_dict_g.values():
        f.write(f"{v}\n")

In [ ]:
filtered_dict_g

In [ ]:
df_dict = pd.DataFrame(list(filtered_dict_g.items()), columns=["KEGG_ID", "GName"])
df_up["KEGG_ID"] = df_up["KEGG_ID"].str.split(":").str[-1]
df_merged = pd.merge(df_up, df_dict, on="KEGG_ID", how="left")
df_merged.head()

In [ ]:
df_merged.to_excel("df_up.xlsx", index=False)

### Downregulated genes

All genes IDs with fold change <= 0.4 (use in the study) and FDR <= 0.05

In [ ]:
df_down = df_all[
    (df_all["C18:2n-6_fold_change"] <= 0.4) &
    (df_all["C18:2n-6_fdr"] <= 0.05)
].copy()
df_down

In [ ]:
downregulted = [x.split(":")[-1] for x in df_down["KEGG_ID"]]
with open('downregulted.txt', 'w') as file:
    for gene in downregulted:
        file.write(f"{gene}\n")
        
len(downregulted)

#### Genes names

In [ ]:
from ResPathExplorer.mapper_KeggFunctions import get_gene_name_by_kegg_id

dict_g = {}
not_found_list = []
for id_g in downregulted:
    name = get_gene_name_by_kegg_id("lmo:" + id_g)
    if name:
        dict_g[id_g] = name.split(",", 1)[0].strip()
    else:
        not_found_list.append(id_g)
print(f"Genes without a KEGG symbol: {len(not_found_list)}")


In [ ]:
filtered_dict_g = {k: v for k, v in dict_g.items() if v != ""}
len(filtered_dict_g)

In [ ]:
with open("downregultedGNames.txt", "w") as f:
    for v in filtered_dict_g.values():
        f.write(f"{v}\n")

In [ ]:
df_dict = pd.DataFrame(list(filtered_dict_g.items()), columns=["KEGG_ID", "GName"])
df_down["KEGG_ID"] = df_down["KEGG_ID"].str.split(":").str[-1]
df_merged = pd.merge(df_down, df_dict, on="KEGG_ID", how="left")
df_merged.head()

In [ ]:
df_merged.to_excel("df_down.xlsx", index=False)